In [1]:
import pandas as pd

df_catmus = pd.read_csv("../dataset_nlp/catmus_french_14_15_clean.csv")
df_himanis = pd.read_csv("../dataset_nlp/himanis_clean.csv")

# Colonnes communes retenues pour le corpus final
common_cols = ['text', 'image_path', 'language', 'century', 'shelfmark', 'project', 'source_corpus']

df_catmus_aligned = df_catmus[common_cols].copy()
df_himanis_aligned = df_himanis[common_cols].copy()

corpus_final = pd.concat([df_catmus_aligned, df_himanis_aligned], ignore_index=True)

# Normalisation : century en string pour la stratification future
corpus_final['century'] = corpus_final['century'].astype(str)

print(f"Total lignes corpus final : {len(corpus_final)}")
print(corpus_final['source_corpus'].value_counts())
print(corpus_final['century'].value_counts())
print(f"Manuscrits/registres distincts (shelfmark) : {corpus_final['shelfmark'].nunique()}")

corpus_final.to_csv("../dataset_nlp/corpus_final_clean.csv", index=False)
print("✅ corpus_final_clean.csv généré.")

Total lignes corpus final : 33510
source_corpus
CATMuS     25812
HIMANIS     7698
Name: count, dtype: int64
century
15    24171
14     9339
Name: count, dtype: int64
Manuscrits/registres distincts (shelfmark) : 40
✅ corpus_final_clean.csv généré.


In [2]:
from sklearn.model_selection import GroupShuffleSplit

# 1ère séparation : 15% pour le test
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_idx, test_idx = next(gss1.split(corpus_final, groups=corpus_final['shelfmark']))

train_val = corpus_final.iloc[train_val_idx]
test = corpus_final.iloc[test_idx]

# 2e séparation sur le reste : on veut encore 15% du TOTAL pour le val
# => proportion à prendre dans train_val (85% du total) = 0.15 / 0.85
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=42)
train_idx, val_idx = next(gss2.split(train_val, groups=train_val['shelfmark']))

train = train_val.iloc[train_idx]
val = train_val.iloc[val_idx]

for name, split in [('train', train), ('val', val), ('test', test)]:
    pct = len(split) / len(corpus_final) * 100
    print(f"\n{name}: {len(split)} lignes ({pct:.1f}%) | groupes: {split['shelfmark'].nunique()}")
    print(split['century'].value_counts(normalize=True).round(3))


train: 26436 lignes (78.9%) | groupes: 28
century
15    0.724
14    0.276
Name: proportion, dtype: float64

val: 3406 lignes (10.2%) | groupes: 6
century
15    0.64
14    0.36
Name: proportion, dtype: float64

test: 3668 lignes (10.9%) | groupes: 6
century
15    0.781
14    0.219
Name: proportion, dtype: float64


In [3]:
import numpy as np

best = None
for seed1 in range(50):
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=seed1)
    train_val_idx, test_idx = next(gss1.split(corpus_final, groups=corpus_final['shelfmark']))
    train_val = corpus_final.iloc[train_val_idx]
    test = corpus_final.iloc[test_idx]

    for seed2 in range(50):
        gss2 = GroupShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=seed2)
        train_idx, val_idx = next(gss2.split(train_val, groups=train_val['shelfmark']))
        train = train_val.iloc[train_idx]
        val = train_val.iloc[val_idx]

        props = np.array([len(train), len(val), len(test)]) / len(corpus_final)
        score = np.sum((props - np.array([0.70, 0.15, 0.15]))**2)

        if best is None or score < best[0]:
            best = (score, seed1, seed2, props, train, val, test)

score, seed1, seed2, props, train, val, test = best
print(f"Meilleurs seeds : gss1={seed1}, gss2={seed2}")
print(f"Proportions : train={props[0]*100:.1f}%, val={props[1]*100:.1f}%, test={props[2]*100:.1f}%")
for name, split in [('train', train), ('val', val), ('test', test)]:
    print(f"\n{name}: {len(split)} lignes | groupes: {split['shelfmark'].nunique()}")
    print(split['century'].value_counts(normalize=True).round(3))

Meilleurs seeds : gss1=44, gss2=24
Proportions : train=70.0%, val=15.6%, test=14.4%

train: 23469 lignes | groupes: 28
century
15    0.657
14    0.343
Name: proportion, dtype: float64

val: 5223 lignes | groupes: 6
century
15    0.802
14    0.198
Name: proportion, dtype: float64

test: 4818 lignes | groupes: 6
century
15    0.947
14    0.053
Name: proportion, dtype: float64


In [4]:
global_century = corpus_final['century'].value_counts(normalize=True)
target_prop = np.array([0.70, 0.15, 0.15])

best = None
for seed1 in range(60):
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=seed1)
    train_val_idx, test_idx = next(gss1.split(corpus_final, groups=corpus_final['shelfmark']))
    train_val = corpus_final.iloc[train_val_idx]
    test = corpus_final.iloc[test_idx]

    for seed2 in range(60):
        gss2 = GroupShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=seed2)
        train_idx, val_idx = next(gss2.split(train_val, groups=train_val['shelfmark']))
        train = train_val.iloc[train_idx]
        val = train_val.iloc[val_idx]

        props = np.array([len(train), len(val), len(test)]) / len(corpus_final)
        score_prop = np.sum((props - target_prop)**2)

        score_century = 0
        for split in (train, val, test):
            sc = split['century'].value_counts(normalize=True)
            for c in ['14', '15']:
                score_century += (sc.get(c, 0) - global_century.get(c, 0))**2

        score = score_prop + score_century
        if best is None or score < best[0]:
            best = (score, seed1, seed2, props, train, val, test)

score, seed1, seed2, props, train, val, test = best
print(f"Meilleurs seeds : gss1={seed1}, gss2={seed2}")
print(f"Proportions : train={props[0]*100:.1f}%, val={props[1]*100:.1f}%, test={props[2]*100:.1f}%")
for name, split in [('train', train), ('val', val), ('test', test)]:
    print(f"\n{name}: {len(split)} lignes | groupes: {split['shelfmark'].nunique()}")
    print(split['century'].value_counts(normalize=True).round(3))

Meilleurs seeds : gss1=47, gss2=57
Proportions : train=70.0%, val=17.8%, test=12.3%

train: 23444 lignes | groupes: 28
century
15    0.724
14    0.276
Name: proportion, dtype: float64

val: 5954 lignes | groupes: 6
century
15    0.731
14    0.269
Name: proportion, dtype: float64

test: 4112 lignes | groupes: 6
century
15    0.695
14    0.305
Name: proportion, dtype: float64


In [5]:
import hashlib
import os

splits_dir = "../dataset_nlp/splits"
os.makedirs(splits_dir, exist_ok=True)

hashes = {}
for name, split in [('train', train), ('val', val), ('test', test)]:
    split_sorted = split.sort_values('image_path').reset_index(drop=True)
    out_path = os.path.join(splits_dir, f"{name}.csv")
    split_sorted.to_csv(out_path, index=False)

    with open(out_path, 'rb') as f:
        h = hashlib.sha256(f.read()).hexdigest()
    hashes[name] = h
    print(f"{name}: {len(split_sorted)} lignes -> {out_path}")
    print(f"  SHA-256: {h}")

with open(os.path.join(splits_dir, "SPLIT_MANIFEST.txt"), "w") as f:
    f.write("Seeds: gss1=47, gss2=57\n")
    f.write("Méthode: GroupShuffleSplit (groupe=shelfmark), 2 passes\n\n")
    for name, h in hashes.items():
        f.write(f"{name}.csv : {h}\n")

print("\n✅ Splits sauvegardés + hashés dans dataset_nlp/splits/")
print("⚠️  À partir de maintenant : plus aucune modification ni inspection de test.csv avant le rendu final.")

train: 23444 lignes -> ../dataset_nlp/splits/train.csv
  SHA-256: 4b30cdb9aece87cac60d835986e0e5bfa331c8c47b1ccd72d473796d882325e3
val: 5954 lignes -> ../dataset_nlp/splits/val.csv
  SHA-256: 592f2e69fb5df7ecb5f7225d012352c32abee860ac13a276cd8bd2159045668e
test: 4112 lignes -> ../dataset_nlp/splits/test.csv
  SHA-256: 3df155b380d8316c29b0f758192fd71dcb9e6f6620e42c090d9f4331cf0a6f08

✅ Splits sauvegardés + hashés dans dataset_nlp/splits/
⚠️  À partir de maintenant : plus aucune modification ni inspection de test.csv avant le rendu final.


In [6]:
import json

entry = {
    "step": "corpus_v1_fondations",
    "date": pd.Timestamp.now().strftime("%Y-%m-%d"),
    "description": (
        "Corpus final reconstruit : CATMuS (25812 lignes, mapping image corrige "
        "via index HF d'origine) + HIMANIS-Guerin JJ207/JJ210 (7698 lignes, "
        "coordonnees PAGE-XML reelles extraites). CREMMA-BS4 (doublon BnF fr.22549, "
        "deja present dans CATMuS) et e-NDP (licence CC-BY-NC-SA non conforme a la "
        "contrainte #7) abandonnes."
    ),
    "total_lines": len(corpus_final),
    "n_groups": int(corpus_final['shelfmark'].nunique()),
    "split": {"train": len(train), "val": len(val), "test": len(test)},
    "split_method": "GroupShuffleSplit x2 (groupe=shelfmark), seeds choisis par recherche multi-objectif (proportions 70/15/15 + equilibre par siecle)",
    "split_seeds": {"gss1": 47, "gss2": 57},
    "century_distribution": {
        "global": corpus_final['century'].value_counts(normalize=True).round(3).to_dict(),
        "train": train['century'].value_counts(normalize=True).round(3).to_dict(),
        "val": val['century'].value_counts(normalize=True).round(3).to_dict(),
        "test": test['century'].value_counts(normalize=True).round(3).to_dict(),
    },
    "sha256": hashes,
    "notes": "Test set scelle a partir de cette etape - ne plus le modifier ni l'inspecter avant le rendu final."
}

with open("../experiments/journal.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print("✅ Entrée ajoutée à experiments/journal.jsonl")

✅ Entrée ajoutée à experiments/journal.jsonl
